# AG_PRAXIS NB03 — Feature Provenance Check

Every attack in this dataset was recorded in its own capture session, and eight of the
nineteen classes were recorded several times over. That arrangement is convenient for
labelling and it creates a problem for measurement. Anything that differs between one
recording and the next, the machine, the link, the load on it, the moment it was made, is
carried in the numbers alongside the attack itself. A model reading those numbers can work
out which recording a row came from, and because a recording holds exactly one attack,
knowing the recording is most of the way to naming the attack without having learned
anything about attacks at all.

This notebook asks how much of that is going on. The blunt version of the worry is already
ruled out: the inventory read every row of every file and found no column that sits at one
value through a whole recording and then moves to a different value in the next one, so no
column is acting as a session label. What is left is quieter. A column that varies inside
every recording but sits at a different level in each one would carry the same information
and never once look constant.

So the test is a model. Take the eight classes that were recorded more than once, throw
away the attack labels, and ask a classifier to name the recording instead. If it can, the
measurements identify the session. Then ask the same question again with the attack already
fixed, one model per class, so the only thing left to tell apart is one DDoS-ICMP recording
from another. That second number is the one that matters, because nothing about it can be
explained away as the model simply telling attacks apart.

Then the same test family by family, and again on the five features published work on this
dataset reports as most important, to see where the signal sits. Then the cost: classifying
the attack with a whole recording held out of training, against classifying it with all the
rows pooled and split at random, to see whether the shortcut is worth anything in accuracy.

Every recording contributes exactly the same number of rows, so no recording can be
identified by being bigger than the others, and chance is one over the number of recordings
rather than something the class sizes decide.

The whole thing is written as one function and run twice. The fast pass uses fewer rows per
recording so that a wrong path or a mistake in the arithmetic surfaces in a couple of
minutes rather than most of an hour, and nothing it produces is a measurement of the
dataset. The full pass is the one that counts. The last cells put the two side by side and
print PASS or MISMATCH for every value that cannot legitimately differ between them, and
the ledger entry refuses to describe itself as a reference run if any of them disagree.

Sixteen models are trained per pass. Each writes `config.json`, `metrics.json`,
`y_true.npy`, `y_pred.npy` and the fitted model. The pass also writes
`variance_ranking.json`, `NB03_verdict.json` and two figures.

The data sits on Drive and the code sits in the repository, so the first block mounts one
and clones the other, and records the commit it is running from. Every number below belongs
to that commit.

In [ ]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

Paths and the seed come from `config/base.yaml`, so changing a path is a commit rather than
an edit to a cell that I later forget I made.

The two passes write to two different folders, `NB03_fast` and `NB03`, so neither can
overwrite the other. `NB03` is the full pass and is the one the repository takes.

In [ ]:
import json
import random
import time

import numpy as np
import pandas as pd
import yaml

from src import captures as cap
from src import inventory as inv
from src.runs import fit_and_save

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
TRAIN_DIR = Path(CFG["paths"]["train_dir"])
TEST_DIR = Path(CFG["paths"]["test_dir"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])
OUT_DIRS = {"fast": ARTIFACTS / "NB03_fast", "full": ARTIFACTS / "NB03"}

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(
        f"{ARTIFACTS} does not exist. Drive is not mounted, or the artefacts path in "
        "config/base.yaml is wrong. Nothing this notebook writes would survive."
    )

ALL_FILES = sorted(TRAIN_DIR.glob("*.csv")) + sorted(TEST_DIR.glob("*.csv"))
if not ALL_FILES:
    raise FileNotFoundError(f"no CSV files under {TRAIN_DIR} or {TEST_DIR}")

pd.set_option("display.max_rows", 400)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

print(f"seed       : {SEED}")
print(f"train dir  : {TRAIN_DIR}   exists={TRAIN_DIR.exists()}")
print(f"test dir   : {TEST_DIR}   exists={TEST_DIR.exists()}")
print(f"files      : {len(ALL_FILES)}")
print(f"fast pass  : {OUT_DIRS['fast']}")
print(f"full pass  : {OUT_DIRS['full']}   <- the one the repository takes")

The column list, the class list and the tier of each class were all settled by the
inventory, so they are read from the file it wrote rather than retyped here. Retyping them
is how two notebooks end up disagreeing about how many classes there are.

One column is dropped. The inventory read every row of every file and found exactly one
column, `Drate`, holding the same value throughout the whole dataset. It separates nothing
from nothing by construction, so it goes and forty-four features remain. `DHCP` stays. An
earlier screen over the opening rows of each file reported that column as constant too, but
the head of a file is not the file, and reading all of it showed `DHCP` moving.

The other thing the inventory settles is the one this notebook depends on most. It found no
column that holds a single value inside a recording and a different single value in the
next, which is the crude form of the problem: a column like that would be a session name
sitting in the feature matrix. There is none. Whatever provenance is in these files has to
be carried some other way, and that is what the rest of this notebook goes looking for.

In [ ]:
INVENTORY_CANDIDATES = [
    REPO_ROOT / "data" / "processed" / "dataset_inventory.json",
    ARTIFACTS / "NB01" / "dataset_inventory.json",
]
INVENTORY_PATH = next((p for p in INVENTORY_CANDIDATES if p.exists()), None)
if INVENTORY_PATH is None:
    raise FileNotFoundError(
        "dataset_inventory.json not found. NB01 has to have been run, and its output moved "
        f"into data/processed/. Looked in: {[str(p) for p in INVENTORY_CANDIDATES]}"
    )

INVENTORY = json.loads(INVENTORY_PATH.read_text())
if INVENTORY.get("is_fast_pass"):
    raise ValueError(f"{INVENTORY_PATH} was written by NB01's fast pass and is not a result")

ALL_COLUMNS = list(INVENTORY["columns"])
CONSTANT_EVERYWHERE = list(INVENTORY["constant_columns"]["constant_everywhere"])
RECORDING_IDENTIFYING = list(INVENTORY["constant_columns"]["recording_identifying"])
DROPPED = ["Drate"]
FEATURES = [c for c in ALL_COLUMNS if c not in DROPPED]

CLASS_INFO = INVENTORY["classes"]
CLASSES = sorted(CLASS_INFO)
TIER = {label: CLASS_INFO[label]["tier"] for label in CLASSES}
GROUP6 = {label: CLASS_INFO[label]["group6"] for label in CLASSES}
TIER_A = sorted(INVENTORY["tiers"]["A"])
ROWS_PER_FILE = INVENTORY["rows_per_file"]
TOTAL_ROWS_NB01 = int(INVENTORY["total_rows"])

print(f"inventory read from {INVENTORY_PATH}")
print(f"  written by  : {INVENTORY['generated_by']} at {INVENTORY['git_sha']} on "
      f"{INVENTORY['generated_on']}")
print(f"  rows scanned: {INVENTORY['rows_scanned']:,}")
print()
print(f"columns in the files                  : {len(ALL_COLUMNS)}")
print(f"constant across the whole scan        : {CONSTANT_EVERYWHERE or 'none'}")
print(f"constant inside a recording, different")
print(f"  between recordings                  : {RECORDING_IDENTIFYING or 'none'}")
print(f"dropped here                          : {DROPPED}")
print(f"features analysed                     : {len(FEATURES)}")
print(f"DHCP retained                         : {'DHCP' in FEATURES}")
print(f"classes with several recordings       : {len(TIER_A)}")
print()

assert len(FEATURES) == 44, f"expected 44 features after the drop, got {len(FEATURES)}"
assert len(CLASSES) == 19, f"expected 19 classes, got {len(CLASSES)}"
if RECORDING_IDENTIFYING:
    print("The inventory found a column that names the recording outright. That changes what")
    print(f"this notebook is testing: {RECORDING_IDENTIFYING}")
else:
    print("No column names the recording outright, so anything found below is carried by")
    print("features that move inside every recording and sit at different levels in each.")

The families come from `config/feature_families.yaml`. That file was written from the
column names alone and is still marked as a draft, so what it gives me is a grouping by
what a column is called rather than by what it measures. It is enough for the question
here, which is whether the signal sits in the four timing columns or is spread across all
forty-four, and the answer is read as a statement about those named groups rather than
about traffic in general.

`Drate` is in the timing family in that file and is dropped here, so timing has four
members. The three families have to cover all forty-four features between them with no
overlap, and that is asserted rather than assumed.

In [ ]:
FAMILIES_PATH = REPO_ROOT / "config" / "feature_families.yaml"
FAMILY_DOC = yaml.safe_load(FAMILIES_PATH.read_text())

FAMILIES = {
    name: [c for c in (members or []) if c in FEATURES]
    for name, members in FAMILY_DOC["families"].items()
}
FAMILIES = {name: members for name, members in FAMILIES.items() if members}
FAMILY_ORDER = [n for n in ("timing", "protocol", "statistical") if n in FAMILIES]
FAMILY_ORDER += [n for n in FAMILIES if n not in FAMILY_ORDER]

assigned = [c for name in FAMILY_ORDER for c in FAMILIES[name]]
missing = [c for c in FEATURES if c not in assigned]
duplicated = sorted({c for c in assigned if assigned.count(c) > 1})

print(f"read {FAMILIES_PATH}")
print(f"  status          : {FAMILY_DOC.get('status')}")
print(f"  review required : {FAMILY_DOC.get('review_required')}")
print()
for name in FAMILY_ORDER:
    print(f"{name:<12} {len(FAMILIES[name]):>2}   {', '.join(FAMILIES[name])}")
print()
print(f"features covered : {len(assigned)} of {len(FEATURES)}")
print(f"unassigned       : {missing or 'none'}")
print(f"in two families  : {duplicated or 'none'}")

assert not missing, f"features with no family: {missing}"
assert not duplicated, f"features in more than one family: {duplicated}"
assert len(assigned) == len(FEATURES), "family sizes do not add up to the feature count"
print()
print("The three families partition the forty-four features, so the family results below")
print("can be read against the all-features result without any column being counted twice.")

Next, what counts as a recording. A recording is one capture file, and the identifier is
the file name with the extension taken off, which means the partition is part of it. That
matters more than it looks. The chunk numbers restart in each partition, so the file called
`TCP_IP-DDoS-ICMP1_test` is not a second helping of `TCP_IP-DDoS-ICMP1_train`; it is a
different session that happens to carry the number one. Identifying recordings by the bare
capture id would quietly merge two separate sessions into one target and make the task
easier than it is.

Only the eight classes recorded more than once are used. For the other eleven there is one
recording each, so asking a model to tell one recording of that class from another has no
answer to give, and asking it to tell those recordings from other classes' recordings is
the attack classification problem wearing a different label.

In [ ]:
def recording_of(path):
    """The recording a file holds: its name without the extension, partition included."""
    meta = cap.parse_capture(Path(path).name)
    return f"{meta['capture_id']}_{meta['partition']}"


rows = []
for path in ALL_FILES:
    meta = cap.parse_capture(path.name)
    if meta["label"] not in TIER_A:
        continue
    rows.append(
        {
            "recording": recording_of(path),
            "label": meta["label"],
            "partition": meta["partition"],
            "rows_available": int(ROWS_PER_FILE[path.name]),
            "path": str(path),
        }
    )

RECORDINGS = pd.DataFrame(rows).sort_values(["label", "recording"]).reset_index(drop=True)
assert pd.api.types.is_numeric_dtype(RECORDINGS["rows_available"]), (
    f"rows_available is {RECORDINGS['rows_available'].dtype}, not numeric"
)
assert RECORDINGS["recording"].is_unique, "two files claim the same recording identifier"

RECORDING_IDS = RECORDINGS["recording"].tolist()
LABEL_OF_RECORDING = dict(zip(RECORDINGS["recording"], RECORDINGS["label"]))
RECORDINGS_PER_CLASS = {
    label: RECORDINGS.loc[RECORDINGS["label"] == label, "recording"].tolist() for label in TIER_A
}
SMALLEST_RECORDING = int(RECORDINGS["rows_available"].min())

print(RECORDINGS[["recording", "label", "partition", "rows_available"]].to_string(index=False))
print()
print(f"recordings                    : {len(RECORDINGS)}")
print(f"classes                       : {len(TIER_A)}   {', '.join(TIER_A)}")
print(f"chance if naming the recording: {1 / len(RECORDINGS):.4f}")
print(f"smallest recording            : {SMALLEST_RECORDING:,} rows")
print()
for label in TIER_A:
    ids = RECORDINGS_PER_CLASS[label]
    print(f"  {label:<12} {len(ids)} recordings, chance {1 / len(ids):.3f}   {', '.join(ids)}")

assert len(RECORDINGS) == 50, f"expected 50 recordings across the eight classes, got {len(RECORDINGS)}"
assert all(len(v) > 1 for v in RECORDINGS_PER_CLASS.values()), "a class here has one recording"

Models are built below, so the seed is set before anything else runs. The same seed draws
the rows, splits them and grows every forest, which is what makes a re-run of this notebook
produce the same numbers rather than nearly the same ones.

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
print(f"seeded with {SEED}")

What follows is one function per step, then a driver that calls them in order, then the
cell that runs the whole thing twice. Every step prints its own results and returns them,
so nothing is recomputed later and the comparison at the end has something to compare.

This cell holds the choices the steps share. The number of rows taken from each recording
is the only thing the fast pass changes. It is the same number for every recording in both
passes, which is the point: if one recording contributed more rows than another the model
could name it by frequency alone and the accuracy would mean nothing.

The forest is the same in every run. Fifty trees rather than a few hundred, and a floor of
a hundred rows per leaf, because the question is whether the recording can be identified at
all rather than what the last point of accuracy would be, and because sixteen fully grown
fifty-class forests written to Drive is several gigabytes of model file for no extra
answer.

The five named features are the ones published work on this dataset reports as carrying the
most weight. Three of them are timing columns, which is why they are also tested on their
own.

The last thing here is what counts as beating chance, and it is not a multiple of it. A
class with five recordings has a chance rate of a fifth, so a model that is right every
single time scores exactly five times chance and one that is right nine times in ten scores
four and a half, which makes a multiple useless for comparing runs that are choosing
between five recordings against runs choosing between fifty. What is used instead is how
much of the distance between chance and perfect a run closes. Half of it or more counts as
identifying the recording, and the same rule reads sensibly whether chance is one in five
or one in fifty.

In [ ]:
ROWS_PER_RECORDING = {"fast": 1_000, "full": 8_000}
FAST_READ_ROWS = 20_000

TEST_FRACTION = 0.30
N_ESTIMATORS = 50
MIN_SAMPLES_LEAF = 100
TOP_FEATURES = 15
TOP_RATIOS = 15

GAP_CLOSED = 0.50

PRIOR_FIVE = ["IAT", "Rate", "Srate", "Header_Length", "rst_count"]
TIMING_THREE = ["IAT", "Rate", "Srate"]

for named in (PRIOR_FIVE, TIMING_THREE):
    absent = [f for f in named if f not in FEATURES]
    assert not absent, f"named features not in the column list: {absent}"

assert ROWS_PER_RECORDING["full"] <= SMALLEST_RECORDING, (
    f"the full pass asks for {ROWS_PER_RECORDING['full']:,} rows per recording and the "
    f"smallest recording holds {SMALLEST_RECORDING:,}"
)


def closes_gap(accuracy, chance):
    """Whether a run closes at least half the distance between chance and perfect."""
    return bool(accuracy >= chance + GAP_CLOSED * (1.0 - chance))


def gap_closed(accuracy, chance):
    """How much of that distance it closed, as a share."""
    return float((accuracy - chance) / (1.0 - chance)) if chance < 1.0 else float("nan")


def section(title):
    print()
    print("-" * 79)
    print(title)
    print("-" * 79)


def banner(lines):
    print()
    print("#" * 79)
    for line in lines:
        print(f"#  {line:<75}#")
    print("#" * 79)


def merge(results, part):
    """Fold a step's return value into the run, keeping figures and runs from every step."""
    results["figures"] = results.get("figures", []) + list(part.pop("figures", []))
    results["runs"] = results.get("runs", []) + list(part.pop("runs", []))
    results.update(part)
    return results


print(f"rows per recording : fast {ROWS_PER_RECORDING['fast']:,}, full "
      f"{ROWS_PER_RECORDING['full']:,}")
print(f"rows in total      : fast {ROWS_PER_RECORDING['fast'] * len(RECORDINGS):,}, full "
      f"{ROWS_PER_RECORDING['full'] * len(RECORDINGS):,}")
print(f"held out for test  : {TEST_FRACTION:.0%} of the rows in every run")
print(f"forest             : {N_ESTIMATORS} trees, at least {MIN_SAMPLES_LEAF} rows per leaf, "
      f"seed {SEED}")
print(f"counts as identified: closing {GAP_CLOSED:.0%} of the distance from chance to 1.0")
print(f"five named features: {', '.join(PRIOR_FIVE)}")
print(f"of which timing    : {', '.join(TIMING_THREE)}")

The two figures share one look, so the styling lives in a single cell rather than being
repeated twice. Figures drawn by the fast pass get a line at the top of the title saying
so, because a figure drawn from fewer rows looks exactly like a result and is not one.

In [ ]:
import matplotlib.pyplot as plt

INK, MUTED, GRID, RULE = "#0b0b0b", "#52514e", "#e6e5e1", "#c9c8c3"
FLAG = "#b03a4a"
BAR = "#3f6fb0"
BAR_ALT = "#d1622b"


def _style(ax, axis="x"):
    ax.grid(axis=axis, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(RULE)
    ax.tick_params(colors=MUTED, length=0)
    for label in ax.get_yticklabels():
        label.set_color(INK)


def _title(text, fast):
    return ("FAST PASS, fewer rows per recording, not a result\n" + text) if fast else text


def _save(fig, path):
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"wrote {path}")
    plt.show()
    plt.close(fig)
    return path


print("figure style set, 300 dpi on save")

Then the rows. Each of the fifty files is read and the same number of rows is drawn from
it at random without replacement, so every recording is represented equally and the draw
covers the whole session rather than its opening minutes. That second part is why the full
pass reads each file in its entirety before sampling: the rows in a capture are in the
order the extractor wrote them, so the head of a file is one contiguous stretch of a
session, and a stretch is exactly the thing whose properties I am trying to measure. Taking
the head would be measuring one part of a recording and calling it the recording.

The fast pass reads only the first twenty thousand rows of each file and draws from those,
which is a head slice and is stated as one. It is there to exercise the code.

The forty-four feature columns are held as float32. The recording identifier and the class
label are held as plain strings and never as a category, because pandas carries a
categorical through a groupby into the result and the arithmetic in the next step would
then fail on a column of counts. Every feature column is asserted numeric here, so a dtype
problem stops with the column named rather than surfacing seven frames later.

In [ ]:
def load_rows(fast):
    mode = "fast" if fast else "full"
    per_recording = ROWS_PER_RECORDING[mode]
    read_cap = FAST_READ_ROWS if fast else None
    scope = (
        f"the first {FAST_READ_ROWS:,} rows of each file"
        if fast
        else "every row of each file"
    )
    section(f"Reading {scope}, then drawing {per_recording:,} rows from each recording")

    rng = np.random.default_rng(SEED)
    frames, taken = [], []
    started = time.time()

    for i, row in enumerate(RECORDINGS.itertuples(), start=1):
        frame = pd.read_csv(row.path, usecols=FEATURES, nrows=read_cap)
        frame = frame[FEATURES]
        wrong = {c: str(t) for c, t in frame.dtypes.items() if not pd.api.types.is_numeric_dtype(t)}
        assert not wrong, f"{Path(row.path).name} has non-numeric feature columns: {wrong}"

        available = len(frame)
        if available < per_recording:
            raise ValueError(
                f"{row.recording} offers {available:,} rows and {per_recording:,} are needed. "
                "Every recording has to contribute the same number of rows or the model can "
                "name a recording by how often it appears."
            )
        chosen = np.sort(rng.choice(available, size=per_recording, replace=False))
        part = frame.iloc[chosen].astype("float32").reset_index(drop=True)
        part["recording"] = row.recording
        part["label"] = row.label
        frames.append(part)
        taken.append(
            {
                "recording": row.recording,
                "label": row.label,
                "rows_read": int(available),
                "rows_taken": int(per_recording),
                "share_of_recording": round(per_recording / available, 4),
            }
        )
        del frame

        if i % 10 == 0 or i == len(RECORDINGS):
            print(f"  {i:>2}/{len(RECORDINGS)} recordings, {time.time() - started:.0f}s")

    data = pd.concat(frames, ignore_index=True)
    del frames

    drawn = pd.DataFrame(taken)
    assert drawn[["rows_read", "rows_taken"]].apply(
        lambda column: pd.api.types.is_numeric_dtype(column)
    ).all(), f"non-numeric count columns: {drawn.dtypes.to_dict()}"

    sizes = data.groupby("recording").size()
    balanced = bool(sizes.nunique() == 1)

    print()
    print(drawn.to_string(index=False))
    print()
    print(f"rows loaded          : {len(data):,}")
    print(f"rows read to get them: {int(drawn['rows_read'].sum()):,}")
    print(f"recordings           : {sizes.size}")
    print(f"rows per recording   : {sorted(sizes.unique().tolist())}")
    print(f"every recording equal: {balanced}")
    print()
    print("rows per class, which are not equal because the classes were recorded a")
    print("different number of times")
    print(data.groupby("label").size().to_string())
    print()

    nonfinite = int((~np.isfinite(data[FEATURES].to_numpy(dtype=np.float32))).sum())
    print(f"values that are not finite : {nonfinite:,}")
    if nonfinite:
        print("A tree splits on comparisons, so a non-finite value stops the fit rather than")
        print("being ignored. They are replaced with zero and the count is reported.")
        data[FEATURES] = data[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    assert balanced, f"recordings contributed different numbers of rows: {sizes.to_dict()}"
    assert data["recording"].dtype == object, "the recording identifier is not a plain string"
    assert data["label"].dtype == object, "the label is not a plain string"

    return {
        "data": data,
        "rows_loaded": int(len(data)),
        "rows_read": int(drawn["rows_read"].sum()),
        "rows_per_recording": int(per_recording),
        "rows_balanced": balanced,
        "rows_drawn": drawn,
        "nonfinite_values": nonfinite,
        "load_s": time.time() - started,
    }

Before any model, a look at the raw numbers. For each feature I take its mean and its
variance inside each recording separately, then compare how far those means sit from one
another against how far the rows sit from their own recording's mean. The first is the
variance between recordings, the second is the average variance within them, and the ratio
of the two says which is larger.

A ratio near zero means the recordings agree: the feature spreads out inside a recording
and every recording is spread around the same place. A ratio near one means the gap between
two recordings is as wide as the spread inside a single one, and a ratio above one means
the recording explains more of the feature's movement than anything happening inside the
recording does. That is a feature describing the session.

Then the same ratio computed one class at a time, and this is the ranking that answers the
question. A recording holds exactly one attack, so a ratio taken across all fifty
recordings at once is measuring two things at the same time: the differences between
attacks, which are supposed to be there, and the differences between recordings of the same
attack, which are not. A feature that separates floods from scans perfectly and never
varies between two recordings of the same flood would score high on the first ranking and
zero on the second. Only the second one is about provenance. The first is printed anyway,
because seeing how far a feature falls between the two is how you tell which of the two
things it was measuring.

The per-class ratios are summarised by their median across the eight classes, with the
largest kept alongside so that a feature that misbehaves in one class only is still
visible.

In [ ]:
def variance_ratio(means, variances, counts):
    """Variance between recordings over mean variance within them, per feature."""
    weights = np.asarray(counts, dtype=float)
    means = np.asarray(means, dtype=float)
    variances = np.asarray(variances, dtype=float)
    grand = np.average(means, axis=0, weights=weights)
    between = np.average((means - grand) ** 2, axis=0, weights=weights)
    within = np.average(variances, axis=0, weights=weights)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(within > 0, between / within, np.nan)
    return between, within, ratio


def measure_variance(data, *, fast, out_dir):
    section("How much each feature moves between recordings against inside one")

    grouped = data.groupby("recording", sort=True)
    means = grouped[FEATURES].mean()
    variances = grouped[FEATURES].var(ddof=0)
    counts = grouped.size()
    order = list(means.index)
    labels = [LABEL_OF_RECORDING[r] for r in order]

    for name, frame in (("means", means), ("variances", variances)):
        wrong = {c: str(t) for c, t in frame.dtypes.items() if not pd.api.types.is_numeric_dtype(t)}
        assert not wrong, f"per-recording {name} came back non-numeric: {wrong}"

    between, within, ratio = variance_ratio(means, variances, counts.reindex(order))
    across = pd.DataFrame(
        {
            "feature": FEATURES,
            "between_recordings": between,
            "within_recordings": within,
            "ratio": ratio,
        }
    ).sort_values("ratio", ascending=False, na_position="last").reset_index(drop=True)
    across.insert(0, "rank", np.arange(1, len(across) + 1))

    print(f"all {len(order)} recordings at once, ranked by the ratio")
    print(
        across.assign(
            between_recordings=lambda f: f["between_recordings"].map(lambda v: f"{v:,.4g}"),
            within_recordings=lambda f: f["within_recordings"].map(lambda v: f"{v:,.4g}"),
            ratio=lambda f: f["ratio"].map(lambda v: f"{v:,.4f}"),
        ).to_string(index=False)
    )
    print()
    print("That ranking mixes two questions. Every recording holds one attack, so a feature")
    print("scores high here either because it separates the attacks or because it separates")
    print("the recordings, and there is no way to tell which from this column. The next one")
    print("holds the attack fixed and only the second explanation survives.")
    print()

    per_class = {}
    for label in TIER_A:
        ids = RECORDINGS_PER_CLASS[label]
        rows = [order.index(r) for r in ids if r in order]
        _, _, class_ratio = variance_ratio(
            means.iloc[rows], variances.iloc[rows], counts.reindex(order).iloc[rows]
        )
        per_class[label] = class_ratio

    within_class = pd.DataFrame(per_class, index=FEATURES)
    summary = pd.DataFrame(
        {
            "feature": FEATURES,
            "median_ratio": within_class.median(axis=1).to_numpy(),
            "max_ratio": within_class.max(axis=1).to_numpy(),
            "worst_class": within_class.idxmax(axis=1).to_numpy(),
            "classes_measured": within_class.notna().sum(axis=1).to_numpy(),
        }
    ).sort_values("median_ratio", ascending=False, na_position="last").reset_index(drop=True)
    summary.insert(0, "rank", np.arange(1, len(summary) + 1))

    print("one class at a time, ranked by the median ratio across the eight classes")
    print(
        summary.assign(
            median_ratio=lambda f: f["median_ratio"].map(lambda v: f"{v:,.4f}"),
            max_ratio=lambda f: f["max_ratio"].map(lambda v: f"{v:,.4f}"),
        ).to_string(index=False)
    )
    print()

    loud = summary[summary["median_ratio"] >= 1.0]
    print(f"features whose median ratio reaches one : {len(loud)} of {len(FEATURES)}")
    if len(loud):
        for row in loud.itertuples():
            print(f"  {row.feature:<16} median {row.median_ratio:.3f}, "
                  f"largest {row.max_ratio:.3f} on {row.worst_class}")
        print()
        print("For these the attack is held fixed and the recordings still sit further apart")
        print("than the rows inside one recording do. Whatever they are measuring includes")
        print("the session.")
    else:
        print("No feature reaches one with the attack held fixed. On this measure the")
        print("recordings of a class agree with each other more closely than the rows inside")
        print("a single recording do.")
    print()

    moved = (
        across[["feature", "rank"]]
        .rename(columns={"rank": "rank_across"})
        .merge(summary[["feature", "rank"]].rename(columns={"rank": "rank_within_class"}),
               on="feature")
    )
    moved["rank_change"] = moved["rank_across"] - moved["rank_within_class"]
    moved = moved.reindex(moved["rank_change"].abs().sort_values(ascending=False).index).head(10)
    print("the ten features that move furthest between the two rankings")
    print(moved.to_string(index=False))
    print()
    print("A feature high in the first ranking and low in the second was separating attacks.")
    print("A feature high in both, or higher in the second, is separating recordings.")

    document = {
        "statistic": "variance of the per-recording means divided by the mean "
                     "within-recording variance",
        "computed_on": f"{len(data):,} rows, {int(counts.iloc[0]):,} from each of "
                       f"{len(order)} recordings",
        "is_fast_pass": bool(fast),
        "across_all_recordings": across.to_dict(orient="records"),
        "within_class": summary.to_dict(orient="records"),
        "per_class_ratios": {
            feature: {
                label: (None if not np.isfinite(v) else round(float(v), 6))
                for label, v in within_class.loc[feature].items()
            }
            for feature in FEATURES
        },
        "features_at_or_above_one_within_class": loud["feature"].tolist(),
        "largest_rank_changes": moved.to_dict(orient="records"),
    }
    path = out_dir / "variance_ranking.json"
    path.write_text(json.dumps(cap.jsonable(document), indent=2, default=str) + "\n")
    print()
    print(f"wrote {path}")

    return {
        "variance_across": across,
        "variance_within_class": summary,
        "variance_per_class": within_class,
        "variance_document": document,
        "variance_path": path,
    }

Now the model. The target is the recording identifier and the attack label is thrown away,
so the only thing the forest can be rewarded for is naming the session a row came from.
Thirty percent of each recording's rows are held out for testing, drawn at random from
within the recording, because every recording has to appear on both sides of the split for
the question to have an answer.

Chance is one over fifty. Anything close to that means the measurements do not carry the
session. Anything far above it means they do, and the size of the gap is the size of the
problem.

The fit and the save are a single statement. Sixteen models are trained across this
notebook and none of them is left sitting in memory unwritten, which is what happens when a
Colab session drops between a fit in one cell and a save in the next.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split


def forest():
    return RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        n_jobs=-1,
        random_state=SEED,
    )


def identify_recording(frame, *, name, features, out_dir, notes, extra_config=None):
    """Fit a forest whose target is the recording, and write the run, in one statement."""
    target = frame["recording"].to_numpy(dtype=object).astype(str)
    X = frame[features].to_numpy(dtype=np.float32)
    X_train, X_test, y_train, y_test = train_test_split(
        X, target, test_size=TEST_FRACTION, random_state=SEED, stratify=target
    )
    return fit_and_save(
        out_dir,
        name,
        forest(),
        X_train,
        y_train,
        X_test,
        y_test,
        features=features,
        target="recording",
        notes=notes,
        extra_config={
            "test_fraction": TEST_FRACTION,
            "n_recordings": int(len(set(target.tolist()))),
            "rows_per_recording": int(len(target) / len(set(target.tolist()))),
            **(extra_config or {}),
        },
        top_k=TOP_FEATURES,
    )


def report(run, *, scope, note=""):
    """One line per run, and the row the figure and the verdict are built from."""
    m = run["metrics"]
    return {
        "name": run["name"],
        "scope": scope,
        "n_features": int(run["config"]["n_features"]),
        "n_classes": int(m["n_classes"]),
        "chance": float(m["chance_rate"]),
        "accuracy": float(m["accuracy"]),
        "macro_f1": float(m["macro_f1"]),
        "weighted_f1": float(m["weighted_f1"]),
        "times_chance": float(m["accuracy"] / m["chance_rate"]),
        "gap_closed": gap_closed(m["accuracy"], m["chance_rate"]),
        "train_seconds": float(m["train_seconds"]),
        "note": note,
    }


def identify_all(data, *, fast, out_dir):
    section("Can a model name the recording, using everything")

    print(f"target      : which of the {len(RECORDINGS)} recordings a row came from")
    print(f"features    : all {len(FEATURES)}")
    print(f"rows        : {len(data):,}, equal from every recording")
    print(f"chance      : {1 / len(RECORDINGS):.4f}")
    print()

    run = identify_recording(
        data,
        name="capture_all_features",
        features=FEATURES,
        out_dir=out_dir,
        notes="which recording a row came from, all 44 features, 8 classes with several "
              "recordings each",
    )
    line = report(run, scope="all 44 features")
    m = run["metrics"]

    print(f"accuracy    : {m['accuracy']:.4f}")
    print(f"macro-F1    : {m['macro_f1']:.4f}")
    print(f"weighted-F1 : {m['weighted_f1']:.4f}")
    print(f"chance      : {m['chance_rate']:.4f}")
    print(f"times chance: {line['times_chance']:,.1f}")
    print(f"gap closed  : {line['gap_closed']:.4f} of the distance from chance to 1.0")
    print(f"trained in  : {m['train_seconds']:.0f}s on {run['config']['n_train']:,} rows")
    print()
    print(f"the {TOP_FEATURES} features it relied on most")
    top = pd.DataFrame(m["top_features"])
    top.insert(0, "rank", np.arange(1, len(top) + 1))
    print(top.assign(importance=lambda f: f["importance"].map(lambda v: f"{v:.4f}")).to_string(
        index=False
    ))
    print()
    if closes_gap(m["accuracy"], m["chance_rate"]):
        print("The recording is identifiable. That does not yet say the model is using")
        print("session properties rather than attack properties, because a recording holds")
        print("one attack and telling the attacks apart already gets it part of the way.")
        print("The next step removes that explanation.")
    else:
        print("The model does little better than guessing, so on this evidence the rows do")
        print("not carry which recording they came from.")

    return {"run_all": run, "runs": [line]}

That result has an innocent reading. Each recording holds one attack, so a model that has
learned nothing except how to tell DDoS-ICMP from DoS-UDP can already narrow fifty
recordings down to the handful belonging to the right class, and score well above chance
without knowing anything about sessions.

This step removes that reading. One model per class, trained only on the recordings of that
class, so the attack is fixed before the model starts and the only question left is whether
one DDoS-ICMP recording can be told from another DDoS-ICMP recording. Chance is now one
over the number of recordings that class has, five or ten rather than fifty.

The mean accuracy across the eight is the number this notebook is really about. Nothing
about it can be explained by the model telling attacks apart, because within each of these
models there is only one attack.

In [ ]:
def identify_within_class(data, *, fast, out_dir):
    section("The same question with the attack already known")

    lines, runs = [], {}
    for label in TIER_A:
        ids = RECORDINGS_PER_CLASS[label]
        part = data[data["label"] == label]
        run = identify_recording(
            part,
            name=f"capture_within_{label.replace('-', '_')}",
            features=FEATURES,
            out_dir=out_dir,
            notes=f"which {label} recording a row came from, attack held fixed",
            extra_config={"class": label},
        )
        runs[label] = run
        lines.append(report(run, scope=f"within {label}", note=f"{len(ids)} recordings"))
        print(f"  {label:<12} {len(ids)} recordings, {len(part):,} rows, "
              f"accuracy {run['metrics']['accuracy']:.4f}")

    table = pd.DataFrame(lines)
    mean_accuracy = float(table["accuracy"].mean())
    mean_macro = float(table["macro_f1"].mean())
    mean_chance = float(table["chance"].mean())

    print()
    print(
        table[["name", "n_classes", "chance", "accuracy", "macro_f1", "gap_closed"]]
        .rename(columns={"n_classes": "recordings"})
        .assign(
            chance=lambda f: f["chance"].map(lambda v: f"{v:.4f}"),
            accuracy=lambda f: f["accuracy"].map(lambda v: f"{v:.4f}"),
            macro_f1=lambda f: f["macro_f1"].map(lambda v: f"{v:.4f}"),
            gap_closed=lambda f: f["gap_closed"].map(lambda v: f"{v:.4f}"),
        )
        .to_string(index=False)
    )
    print()
    print(f"mean accuracy across the {len(TIER_A)} classes : {mean_accuracy:.4f}")
    print(f"mean macro-F1                        : {mean_macro:.4f}")
    print(f"mean chance                          : {mean_chance:.4f}")
    print(f"gap closed, chance to 1.0            : "
          f"{gap_closed(mean_accuracy, mean_chance):.4f}")
    print()
    if closes_gap(mean_accuracy, mean_chance):
        print("With the attack fixed the recording is still identifiable, so the features are")
        print("carrying something that changes from one session to the next and is not the")
        print("attack. Any model trained on these columns can read that, and since a")
        print("recording holds one class, reading it is a route to the label that has")
        print("nothing to do with the traffic.")
    else:
        print("With the attack fixed the model is close to guessing, so the earlier result")
        print("was the model telling attacks apart rather than telling sessions apart.")

    return {
        "runs_within_class": runs,
        "within_class_table": table,
        "within_class_mean_accuracy": mean_accuracy,
        "within_class_mean_macro_f1": mean_macro,
        "within_class_mean_chance": mean_chance,
        "runs": lines,
    }

If the recording can be read off the features, the next thing to know is which features.
The same fifty-way question again, three more times, each with only one family of columns
available. Timing has four members, protocol twenty-eight and statistical twelve.

The comparison to keep in mind is the all-features result. A family that reaches nearly the
same accuracy on a handful of columns is where the provenance sits, and dropping it would
be a real intervention. A family that stays near chance is not carrying it, whatever else
it is good for.

In [ ]:
def identify_by_family(data, *, fast, out_dir):
    section("Which family of features carries it")

    lines, runs = [], {}
    for family in FAMILY_ORDER:
        members = FAMILIES[family]
        run = identify_recording(
            data,
            name=f"capture_family_{family}",
            features=members,
            out_dir=out_dir,
            notes=f"which recording a row came from, {family} family only",
            extra_config={"family": family},
        )
        runs[family] = run
        lines.append(report(run, scope=f"{family} family", note=f"{len(members)} features"))
        print(f"  {family:<12} {len(members):>2} features, "
              f"accuracy {run['metrics']['accuracy']:.4f}")

    table = pd.DataFrame(lines)
    print()
    print(
        table[["scope", "n_features", "accuracy", "macro_f1", "times_chance"]]
        .assign(
            accuracy=lambda f: f["accuracy"].map(lambda v: f"{v:.4f}"),
            macro_f1=lambda f: f["macro_f1"].map(lambda v: f"{v:.4f}"),
            times_chance=lambda f: f["times_chance"].map(lambda v: f"{v:,.1f}"),
        )
        .to_string(index=False)
    )
    print()
    best = table.loc[table["accuracy"].idxmax()]
    print(f"the strongest family is {best['scope']} at {best['accuracy']:.4f} on "
          f"{int(best['n_features'])} features")
    print("Read that against the all-features number rather than against chance. A family")
    print("that reaches nearly the same accuracy on a fraction of the columns is where the")
    print("session is being carried.")

    return {
        "runs_family": runs,
        "family_table": table,
        "strongest_family": str(best["scope"]),
        "runs": lines,
    }

The families are a grouping by name. This step is narrower and more pointed. Published work
on this dataset reports five features as carrying the most weight in its models: `IAT`,
`Rate`, `Srate`, `Header_Length` and `rst_count`. If those five can name the recording, then
the features a detector is reported to depend on are the same features that describe the
session, and a feature ranking taken from such a model is partly a ranking of recording
conditions.

Three of the five are timing columns, so they are run again on their own. Timing separating
attacks is expected, since a flood and a scan differ in how fast packets arrive. Timing
separating two recordings of the same attack is not, because the attack is identical in
both, and whatever is left is the machine, the link and the load.

In [ ]:
def identify_named_features(data, *, fast, out_dir):
    section("The five features published work reports as most important")

    lines, runs = [], {}
    for name, members, note in (
        ("capture_prior_five", PRIOR_FIVE, "the five named by published work"),
        ("capture_timing_three", TIMING_THREE, "the three timing features among them"),
    ):
        run = identify_recording(
            data,
            name=name,
            features=members,
            out_dir=out_dir,
            notes=f"which recording a row came from, {note}",
            extra_config={"feature_set": note},
        )
        runs[name] = run
        lines.append(report(run, scope=note, note=", ".join(members)))
        print(f"  {note:<36} {len(members)} features, "
              f"accuracy {run['metrics']['accuracy']:.4f}")

    table = pd.DataFrame(lines)
    print()
    print(
        table[["scope", "n_features", "accuracy", "macro_f1", "times_chance"]]
        .assign(
            accuracy=lambda f: f["accuracy"].map(lambda v: f"{v:.4f}"),
            macro_f1=lambda f: f["macro_f1"].map(lambda v: f"{v:.4f}"),
            times_chance=lambda f: f["times_chance"].map(lambda v: f"{v:,.1f}"),
        )
        .to_string(index=False)
    )
    print()
    print("features in each set")
    for line in lines:
        print(f"  {line['scope']:<36} {line['note']}")

    return {"runs_named": runs, "named_table": table, "runs": lines}

Everything so far says what a model can read. This step asks what it is worth.

The same eight classes and the same forest, but the target is the attack rather than the
recording, and the split is done two ways. The first holds a whole recording of each class
out of training and tests on that, so the model has never seen the session it is scored on.
The second pools every row and splits at random, so the training rows and the test rows come
from the same sessions.

The two test sets are the same size and the two training sets have the same number of rows,
so the only difference is whether the test rows come from a recording the model has already
seen. If provenance is being used, the pooled split is the one that benefits, and the gap
between the two macro-F1 scores is what the shortcut is worth in the units the project
reports its results in.

The recording held out of each class is the first one from the test partition. The chunk
numbers restart between partitions, so that file is a genuinely separate session rather
than a later slice of a training one.

In [ ]:
def classify_attack(data, *, fast, out_dir):
    section("What the shortcut is worth")

    holdout = {}
    for label in TIER_A:
        ids = RECORDINGS_PER_CLASS[label]
        test_side = [r for r in ids if r.endswith("_test")]
        holdout[label] = (test_side or ids)[-1]

    held = set(holdout.values())
    print("recording held out of training, one per class")
    for label in TIER_A:
        print(f"  {label:<12} {holdout[label]}")
    print()

    labels = data["label"].to_numpy(dtype=object).astype(str)
    X = data[FEATURES].to_numpy(dtype=np.float32)
    is_held = data["recording"].isin(held).to_numpy()

    print(f"held-out split : train {int((~is_held).sum()):,} rows, "
          f"test {int(is_held.sum()):,} rows")

    held_run = fit_and_save(
        out_dir,
        "attack_capture_held_out",
        forest(),
        X[~is_held],
        labels[~is_held],
        X[is_held],
        labels[is_held],
        features=FEATURES,
        target="label",
        notes="attack class, one whole recording per class held out of training",
        extra_config={"protocol": "capture held out", "held_out": holdout},
        top_k=TOP_FEATURES,
    )

    print(f"pooled split   : the same {len(data):,} rows split at random, "
          f"test {int(is_held.sum()):,} rows")

    X_train, X_test, y_train, y_test = train_test_split(
        X, labels, test_size=int(is_held.sum()), random_state=SEED, stratify=labels
    )
    pooled_run = fit_and_save(
        out_dir,
        "attack_rows_pooled",
        forest(),
        X_train,
        y_train,
        X_test,
        y_test,
        features=FEATURES,
        target="label",
        notes="attack class, all rows pooled and split at random",
        extra_config={"protocol": "rows pooled"},
        top_k=TOP_FEATURES,
    )

    held_f1 = float(held_run["metrics"]["macro_f1"])
    pooled_f1 = float(pooled_run["metrics"]["macro_f1"])
    difference = pooled_f1 - held_f1

    print()
    print(f"{'protocol':<24} {'accuracy':>10} {'macro-F1':>10} {'weighted-F1':>12}")
    print("-" * 60)
    for name, run in (("whole recording held out", held_run), ("rows pooled at random", pooled_run)):
        m = run["metrics"]
        print(f"{name:<24} {m['accuracy']:>10.4f} {m['macro_f1']:>10.4f} "
              f"{m['weighted_f1']:>12.4f}")
    print("-" * 60)
    print(f"{'difference, pooled minus held out':<24} {'':>10} {difference:>10.4f}")
    print()
    if abs(difference) < 0.01:
        print("The two protocols land in the same place. The recording is identifiable and")
        print("the model does not need it: these eight classes are far enough apart that")
        print("nothing is left for a shortcut to add. That is a statement about these eight")
        print("classes, which are the volumetric floods, and not about the classes that are")
        print("hard to separate.")
    else:
        print("The two protocols disagree by more than a point of macro-F1, so part of what")
        print("the pooled score measures is the model recognising sessions it has already")
        print("seen rather than attacks it has learned.")

    return {
        "run_attack_held_out": held_run,
        "run_attack_pooled": pooled_run,
        "attack_holdout": holdout,
        "attack_macro_f1_held_out": held_f1,
        "attack_macro_f1_pooled": pooled_f1,
        "attack_macro_f1_difference": difference,
    }

Two figures. The first puts every capture-identification accuracy on one axis with the
chance rate for each drawn beside it, so the fifty-way runs and the within-class runs can be
read together even though they are not guessing against the same number of recordings. The
dashed line is one over fifty, which is the chance rate for every run that had all fifty
recordings to choose from; the marks are the chance rate for each of the eight within-class
runs, where the choice is between five or ten.

The second is the between-against-within variance ratio for the fifteen features that rank
highest with the attack held fixed. The axis is logarithmic where the ratios span more than
two orders of magnitude, because otherwise one feature at fifty flattens the other fourteen
into the spine.

In [ ]:
def draw_figures(r, *, fast, out_dir):
    section("The figures")

    table = pd.DataFrame(r["runs"]).sort_values("accuracy").reset_index(drop=True)
    overall_chance = 1 / len(RECORDINGS)

    fig, ax = plt.subplots(figsize=(11, 8))
    colours = [BAR_ALT if row.scope.startswith("within ") else BAR for row in table.itertuples()]
    bars = ax.barh(table["scope"], table["accuracy"], color=colours, height=0.68)
    ax.bar_label(
        bars,
        labels=[f"{v:.3f}" for v in table["accuracy"]],
        padding=4,
        color=MUTED,
        fontsize=8,
    )
    ax.axvline(overall_chance, color=FLAG, linewidth=1, linestyle="--")
    ax.text(
        overall_chance,
        len(table) - 0.3,
        f"  chance across all {len(RECORDINGS)} recordings, {overall_chance:.3f}",
        color=FLAG,
        fontsize=8,
        va="top",
    )
    for i, row in enumerate(table.itertuples()):
        if abs(row.chance - overall_chance) > 1e-9:
            ax.plot([row.chance], [i], marker="|", markersize=12, color=FLAG)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("accuracy at naming the recording a row came from", color=MUTED, fontsize=9)
    ax.set_title(
        _title(
            f"Every capture-identification run, against chance. Marks show the {len(TIER_A)} "
            "runs with their own chance rate",
            fast,
        ),
        color=INK,
        fontsize=11,
        loc="left",
        pad=12,
    )
    _style(ax)
    ax.tick_params(axis="y", labelsize=8)
    accuracy_path = _save(fig, out_dir / "NB03_capture_identification.png")

    top = r["variance_within_class"].head(TOP_RATIOS).iloc[::-1]
    values = top["median_ratio"].to_numpy(dtype=float)
    positive = values[np.isfinite(values) & (values > 0)]
    log_axis = bool(positive.size and positive.max() / max(positive.min(), 1e-12) > 100)

    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.barh(top["feature"], top["median_ratio"], color=BAR, height=0.68)
    ax.bar_label(
        bars,
        labels=[f"{v:,.3g}" for v in top["median_ratio"]],
        padding=4,
        color=MUTED,
        fontsize=8,
    )
    ax.axvline(1.0, color=FLAG, linewidth=1, linestyle="--")
    if log_axis:
        ax.set_xscale("log")
    ax.set_xlabel(
        "variance between recordings over mean variance within them, median across the "
        f"{len(TIER_A)} classes" + (", log axis" if log_axis else ""),
        color=MUTED,
        fontsize=9,
    )
    ax.set_title(
        _title(
            f"The {TOP_RATIOS} features that shift most between recordings of the same "
            "attack. The line is one",
            fast,
        ),
        color=INK,
        fontsize=11,
        loc="left",
        pad=12,
    )
    _style(ax)
    ax.tick_params(axis="y", labelsize=9)
    ratio_path = _save(fig, out_dir / "NB03_variance_ratio.png")

    return {"figures": [accuracy_path, ratio_path]}

Then the verdict, written out in sentences rather than left for a reader to assemble from
four tables. Three things have to come out of it: whether the recording can be identified at
all, which family of features carries that, and what holding a whole recording out of
training cost in macro-F1.

The thresholds are stated in the cell rather than argued over afterwards. A run counts as
identifying the recording if it closes at least half the distance between its own chance
rate and being right every time, and the within-class runs are the ones that decide it,
since they are the only ones where telling attacks apart is not an available explanation. A
family counts as carrying the signal if it gets within five points of accuracy of using all
forty-four features.

The same text goes into `NB03_verdict.json` alongside the numbers it was derived from, so a
later notebook can act on the decision without reading this one.

In [ ]:
FAMILY_WITHIN = 0.05


def write_verdict(r, *, fast, out_dir):
    section("The verdict")

    table = pd.DataFrame(r["runs"])
    all_features = table[table["name"] == "capture_all_features"].iloc[0]
    within_mean = r["within_class_mean_accuracy"]
    within_chance = r["within_class_mean_chance"]
    families = table[table["name"].str.startswith("capture_family_")]
    named = table[table["name"].isin(["capture_prior_five", "capture_timing_three"])]

    identified = closes_gap(within_mean, within_chance)
    carriers = families[families["accuracy"] >= all_features["accuracy"] - FAMILY_WITHIN]
    carrier_names = carriers.sort_values("accuracy", ascending=False)["scope"].tolist()
    strongest = families.loc[families["accuracy"].idxmax()]
    difference = r["attack_macro_f1_difference"]
    top_ratio = r["variance_within_class"].iloc[0]

    if identified:
        first = (
            f"The recording can be identified. With the attack held fixed, so that telling "
            f"one attack from another is no help, a forest names which of a class's "
            f"recordings a row came from with mean accuracy {within_mean:.4f} against a "
            f"chance rate of {within_chance:.4f}. Across all {len(RECORDINGS)} recordings at "
            f"once the accuracy is {all_features['accuracy']:.4f} against "
            f"{all_features['chance']:.4f}."
        )
    else:
        first = (
            f"The recording cannot be identified once the attack is held fixed. Mean accuracy "
            f"across the {len(TIER_A)} within-class runs is {within_mean:.4f} against a chance "
            f"rate of {within_chance:.4f}, and the higher figure of "
            f"{all_features['accuracy']:.4f} across all {len(RECORDINGS)} recordings is the "
            f"model telling attacks apart rather than sessions."
        )

    second = (
        f"The strongest family on its own is the {strongest['scope']}: "
        f"{int(strongest['n_features'])} of the {len(FEATURES)} features reach accuracy "
        f"{strongest['accuracy']:.4f}, against {all_features['accuracy']:.4f} for all "
        f"{len(FEATURES)}. "
        + (
            f"Families that come within {FAMILY_WITHIN:.2f} of using everything, and so "
            f"carry the signal on their own: {', '.join(carrier_names)}."
            if len(carriers)
            else f"No family comes within {FAMILY_WITHIN:.2f} of using everything, so no one "
                 "family carries it by itself."
        )
        + f" The three timing features named by published work reach "
        f"{named[named['name'] == 'capture_timing_three'].iloc[0]['accuracy']:.4f} between "
        f"them, and all five named features reach "
        f"{named[named['name'] == 'capture_prior_five'].iloc[0]['accuracy']:.4f}."
    )

    third = (
        f"Holding a whole recording out of training cost {abs(difference):.4f} macro-F1 on the "
        f"attack classification: {r['attack_macro_f1_held_out']:.4f} with a recording held "
        f"out against {r['attack_macro_f1_pooled']:.4f} with the rows pooled and split at "
        f"random. "
        + (
            "The two protocols agree, so on these eight classes the identifiable session is "
            "not being used to score points. It is still there to be used by any model "
            "trained on these columns, and by any explanation read off one."
            if abs(difference) < 0.01
            else "Part of the pooled score is the model recognising sessions rather than "
                 "attacks."
        )
    )

    fourth = (
        f"The feature that shifts most between recordings of the same attack is "
        f"{top_ratio['feature']}, at a median ratio of {top_ratio['median_ratio']:,.3f} and "
        f"{top_ratio['max_ratio']:,.3f} on {top_ratio['worst_class']}. No column is constant "
        f"inside a recording, so nothing here is a session label sitting in a column; it is "
        f"carried by where each feature sits."
    )

    consequence = (
        (
            f"Feature attributions taken from a model that can see the {strongest['scope']} "
            f"describe recording conditions as much as attack behaviour, so the explanation "
            f"stage runs on a model with that family excluded."
        )
        if identified
        else "No restriction on the explanation stage follows from this notebook, because "
             "the recording could not be identified with the attack held fixed."
    )

    document = {
        "generated_by": "AG_PRAXIS_NB03_feature_provenance.ipynb",
        "generated_on": RUN_DATE,
        "git_sha": GIT_SHA,
        "git_dirty": GIT_DIRTY,
        "seed": SEED,
        "mode": r["mode"],
        "is_fast_pass": bool(fast),
        "enterable_in_ledger": not fast,
        "source_inventory": str(INVENTORY_PATH),
        "recording_identifiable": identified,
        "threshold": f"closes at least {GAP_CLOSED:.0%} of the distance between the chance "
                     "rate and being right every time, measured on the within-class runs",
        "within_class_gap_closed": gap_closed(within_mean, within_chance),
        "verdict": [first, second, third, fourth],
        "consequence": consequence,
        "rows_loaded": r["rows_loaded"],
        "rows_per_recording": r["rows_per_recording"],
        "rows_balanced": r["rows_balanced"],
        "n_recordings": len(RECORDINGS),
        "recordings": RECORDING_IDS,
        "classes": TIER_A,
        "recordings_per_class": {k: len(v) for k, v in RECORDINGS_PER_CLASS.items()},
        "n_features": len(FEATURES),
        "features": FEATURES,
        "families": {name: FAMILIES[name] for name in FAMILY_ORDER},
        "forest": {
            "n_estimators": N_ESTIMATORS,
            "min_samples_leaf": MIN_SAMPLES_LEAF,
            "test_fraction": TEST_FRACTION,
            "random_state": SEED,
        },
        "capture_identification": table.to_dict(orient="records"),
        "within_class_mean_accuracy": within_mean,
        "within_class_mean_macro_f1": r["within_class_mean_macro_f1"],
        "within_class_mean_chance": within_chance,
        "strongest_family": str(strongest["scope"]),
        "families_within_threshold": carrier_names,
        "attack_classification": {
            "held_out_recording_per_class": r["attack_holdout"],
            "macro_f1_capture_held_out": r["attack_macro_f1_held_out"],
            "macro_f1_rows_pooled": r["attack_macro_f1_pooled"],
            "difference_pooled_minus_held_out": difference,
        },
        "top_variance_ratios": r["variance_within_class"].head(TOP_RATIOS).to_dict(
            orient="records"
        ),
        "variance_ranking_file": "variance_ranking.json",
        "figures": [p.name for p in r["figures"]],
    }

    for line in document["verdict"]:
        print(line)
        print()
    print(consequence)

    path = out_dir / "NB03_verdict.json"
    path.write_text(json.dumps(cap.jsonable(document), indent=2, default=str) + "\n")
    print()
    print(f"wrote {path}")

    return {
        "verdict_document": document,
        "verdict_path": path,
        "recording_identifiable": identified,
    }

`run_provenance` calls the steps in order and hands each one what the previous ones
produced. It takes a single argument, and the only thing that argument changes is how many
rows are drawn from each recording. Nothing is computed a different way in fast mode, and
every model in both passes is the same forest with the same seed.

The loaded rows are dropped from the returned result once the steps that need them have
run, so two passes' worth of data are never in memory at the same time. Everything computed
from them is kept.

In [ ]:
def run_provenance(fast: bool) -> dict:
    mode = "fast" if fast else "full"
    out_dir = OUT_DIRS[mode]
    runs_dir = out_dir / "runs"
    runs_dir.mkdir(parents=True, exist_ok=True)

    started = time.time()
    r = {"mode": mode, "fast": fast, "out_dir": out_dir, "runs_dir": runs_dir,
         "figures": [], "runs": []}

    merge(r, load_rows(fast))
    merge(r, measure_variance(r["data"], fast=fast, out_dir=out_dir))
    merge(r, identify_all(r["data"], fast=fast, out_dir=runs_dir))
    merge(r, identify_within_class(r["data"], fast=fast, out_dir=runs_dir))
    merge(r, identify_by_family(r["data"], fast=fast, out_dir=runs_dir))
    merge(r, identify_named_features(r["data"], fast=fast, out_dir=runs_dir))
    merge(r, classify_attack(r["data"], fast=fast, out_dir=runs_dir))
    merge(r, draw_figures(r, fast=fast, out_dir=out_dir))
    merge(r, write_verdict(r, fast=fast, out_dir=out_dir))

    r.pop("data")

    section(f"What the {mode} pass wrote to {out_dir}")
    r["run_names"] = [line["name"] for line in r["runs"]] + [
        "attack_capture_held_out",
        "attack_rows_pooled",
    ]
    for name in r["run_names"]:
        files = sorted(p.name for p in (runs_dir / name).glob("*"))
        print(f"  runs/{name:<32} {', '.join(files)}")
    print(f"  {r['variance_path'].name}")
    print(f"  {r['verdict_path'].name}")
    for path in r["figures"]:
        print(f"  {path.name}")
    if IN_COLAB:
        print()
        print("Colab cannot push to the repository from a cell. Download these from Drive and")
        print("move them in from the Mac.")

    r["elapsed_s"] = time.time() - started
    return r

Both passes run here, fast first. If the fast one raises, the full one never starts, which
is the point: a wrong path or a mistake in the arithmetic costs a couple of minutes rather
than the best part of an hour.

In [ ]:
BANNERS = {
    "fast": [
        "FAST PASS",
        f"every step, but {ROWS_PER_RECORDING['fast']:,} rows per recording drawn from the",
        "head of each file, so a test of the code and not of the data",
        "never entered in the ledger",
    ],
    "full": [
        "FULL PASS",
        f"every step, {ROWS_PER_RECORDING['full']:,} rows drawn from the whole of each",
        "recording",
        "this is the pass the repository takes",
    ],
}

results = {}
for fast in (True, False):
    name = "fast" if fast else "full"
    banner(BANNERS[name])
    results[name] = run_provenance(fast)

FAST, FULL = results["fast"], results["full"]
banner([f"fast pass {FAST['elapsed_s']:.0f}s, full pass {FULL['elapsed_s']:.0f}s"])

The full pass's two documents, printed whole. Colab cannot push to the repository from a
cell, so until the artefacts are moved across by hand the saved copy of this notebook is the
only durable record of what the run produced. Printing the files means the executed notebook
carries the result even if the Drive folder is later cleared.

Only the full pass is printed. The fast pass's copies are under `NB03_fast` if they are ever
wanted for debugging, and they are not results.

In [ ]:
for path in (FULL["variance_path"], FULL["verdict_path"]):
    print("=" * 79)
    print(f"{path}   ({path.stat().st_size:,} bytes)")
    print("=" * 79)
    print(path.read_text().rstrip())
    print()

print("=" * 79)
print("the sixteen runs, each holding config.json, metrics.json, y_true.npy, y_pred.npy")
print("and the fitted model")
print("=" * 79)
for name in FULL["run_names"]:
    run_dir = FULL["runs_dir"] / name
    metrics = json.loads((run_dir / "metrics.json").read_text())
    size = sum(p.stat().st_size for p in run_dir.glob("*"))
    print(f"{name:<34} accuracy {metrics['accuracy']:.4f}  macro-F1 "
          f"{metrics['macro_f1']:.4f}  {size / 1e6:,.1f} MB")
print()

print("=" * 79)
print("figures, which cannot be printed")
print("=" * 79)
for path in FULL["figures"]:
    print(f"{path.name}   {path.stat().st_size:,} bytes")

Now the two passes side by side.

Only some of what this notebook produces can be compared. Which files are recordings, how
they group into classes, which features are in which family, which feature set each model
was given, what chance is for each model, which recording was held out of training, and
whether every recording contributed the same number of rows are all fixed before any data is
read. They cannot legitimately differ, so a MISMATCH on any of them is a bug in this
notebook rather than something learned about the dataset, and nothing from a run with a
mismatch should be reported.

Everything else is the point of running twice. Every accuracy, every variance ratio and both
macro-F1 scores are computed from a different number of rows in each pass and are supposed
to differ. They are listed underneath with what each pass produced, so the size of the gap
between a cheap answer and a measured one is visible rather than assumed.

In [ ]:
def signature(value, ndigits=10):
    """A comparable form: frames flattened, floats rounded, NaN made equal to itself."""
    if isinstance(value, pd.DataFrame):
        return signature(value.to_dict(orient="records"), ndigits)
    if isinstance(value, pd.Series):
        return signature(value.to_list(), ndigits)
    if isinstance(value, dict):
        return {str(k): signature(v, ndigits) for k, v in value.items()}
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return [signature(v, ndigits) for v in value]
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (float, np.floating)):
        return "nan" if not np.isfinite(value) else round(float(value), ndigits)
    if isinstance(value, (int, np.integer)):
        return int(value)
    return value


def run_config(r, key):
    return {
        name: signature(json.loads((r["runs_dir"] / name / "config.json").read_text())[key])
        for name in r["run_names"]
    }


COMPARISONS = [
    ("features analysed", lambda r: sorted(r["variance_across"]["feature"].tolist())),
    ("recordings found", lambda r: sorted(r["rows_drawn"]["recording"])),
    ("classes used", lambda r: sorted(set(r["rows_drawn"]["label"]))),
    ("recordings per class", lambda r: r["rows_drawn"].groupby("label").size().to_dict()),
    ("every recording equal in size", lambda r: r["rows_balanced"]),
    ("rows taken per recording equal", lambda r: sorted(set(r["rows_drawn"]["rows_taken"]))
     == [r["rows_per_recording"]]),
    ("models trained", lambda r: r["run_names"]),
    ("features given to each model", lambda r: run_config(r, "features")),
    ("feature count per model", lambda r: run_config(r, "n_features")),
    ("target of each model", lambda r: run_config(r, "target")),
    ("forest parameters", lambda r: run_config(r, "params")),
    ("classes each model chose between", lambda r: {
        line["name"]: line["n_classes"] for line in r["runs"]}),
    ("chance rate per model", lambda r: {
        line["name"]: signature(line["chance"]) for line in r["runs"]}),
    ("recording held out per class", lambda r: r["attack_holdout"]),
    ("feature families", lambda r: {name: FAMILIES[name] for name in FAMILY_ORDER}),
]

mismatches = []
print(f"{'value':<34} {'result':<10} fast vs full")
print("-" * 79)
for label, get in COMPARISONS:
    a, b = signature(get(FAST)), signature(get(FULL))
    ok = a == b
    print(f"{label:<34} {'PASS' if ok else 'MISMATCH':<10} " + ("" if ok else "see below"))
    if not ok:
        mismatches.append((label, a, b))

print("-" * 79)
print(f"{len(COMPARISONS) - len(mismatches)} of {len(COMPARISONS)} values agree")
COMPARISON_OK = not mismatches

if mismatches:
    print()
    print("The fast path and the full path disagree on something that cannot legitimately")
    print("differ. That is a bug in this notebook, not a finding. Nothing from this run")
    print("should be reported.")
    for label, a, b in mismatches:
        print()
        print(f"  {label}")
        print(f"    fast : {str(a)[:400]}")
        print(f"    full : {str(b)[:400]}")

print()
print("not compared, because these are the values the two passes are supposed to disagree on")
print(f"  the fast pass drew {FAST['rows_per_recording']:,} rows per recording, the full pass "
      f"{FULL['rows_per_recording']:,}")
print(f"  the fast pass read {FAST['rows_read']:,} rows to get them, the full pass "
      f"{FULL['rows_read']:,}")
print()
print(f"{'step':<40} {'fast':<20} full")
print("-" * 79)
fast_runs = {line["name"]: line for line in FAST["runs"]}
full_runs = {line["name"]: line for line in FULL["runs"]}
for name, label in (
    ("capture_all_features", "capture identification, all features"),
    ("capture_family_timing", "capture identification, timing only"),
    ("capture_family_protocol", "capture identification, protocol only"),
    ("capture_family_statistical", "capture identification, statistical only"),
    ("capture_prior_five", "capture identification, five named"),
    ("capture_timing_three", "capture identification, three timing"),
):
    if name in fast_runs and name in full_runs:
        print(f"{label:<40} {fast_runs[name]['accuracy']:<20.4f} {full_runs[name]['accuracy']:.4f}")
for label, a, b in (
    (
        "capture identification, attack fixed",
        f"{FAST['within_class_mean_accuracy']:.4f}",
        f"{FULL['within_class_mean_accuracy']:.4f}",
    ),
    (
        "top feature by within-class ratio",
        f"{FAST['variance_within_class'].iloc[0]['feature']} "
        f"{FAST['variance_within_class'].iloc[0]['median_ratio']:,.3g}",
        f"{FULL['variance_within_class'].iloc[0]['feature']} "
        f"{FULL['variance_within_class'].iloc[0]['median_ratio']:,.3g}",
    ),
    (
        "attack macro-F1, recording held out",
        f"{FAST['attack_macro_f1_held_out']:.4f}",
        f"{FULL['attack_macro_f1_held_out']:.4f}",
    ),
    (
        "attack macro-F1, rows pooled",
        f"{FAST['attack_macro_f1_pooled']:.4f}",
        f"{FULL['attack_macro_f1_pooled']:.4f}",
    ),
    (
        "recording identifiable",
        f"{FAST['recording_identifiable']}",
        f"{FULL['recording_identifiable']}",
    ),
):
    print(f"{label:<40} {a:<20} {b}")
print()
print("Where those two columns are close, fewer rows happened to be enough for that")
print("quantity. That is worth knowing and it is not a justification for the fast pass: it")
print("is only visible because the full pass was run.")

The entry for `RESULTS_LEDGER.md`, ready to paste. It reports the full pass only. If the two
passes disagreed on anything they should have agreed on, the entry says so at the top and
the run is not a reference run.

In [ ]:
if not COMPARISON_OK:
    status = "DO NOT ENTER, the fast and full passes disagree and the notebook has a bug"
elif FULL["rows_per_recording"] != ROWS_PER_RECORDING["full"]:
    status = (
        f"DO NOT ENTER, the full pass drew {FULL['rows_per_recording']:,} rows per recording "
        f"and {ROWS_PER_RECORDING['full']:,} were asked for"
    )
elif not FULL["rows_balanced"]:
    status = "DO NOT ENTER, the recordings did not contribute equal numbers of rows"
elif GIT_DIRTY:
    status = "reference run, working tree dirty"
else:
    status = "reference run"

lines = {line["name"]: line for line in FULL["runs"]}
all_features = lines["capture_all_features"]
verdict = FULL["verdict_document"]
top_ratio = FULL["variance_within_class"].iloc[0]

ledger = f'''
### NB03 — feature provenance check ({RUN_DATE})

| field | value |
|---|---|
| notebook | AG_PRAXIS_NB03_feature_provenance.ipynb |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SEED} |
| pass reported | full, {FULL["rows_per_recording"]:,} rows drawn from the whole of each recording |
| fast/full agreement | {"all {} compared values agree".format(len(COMPARISONS)) if COMPARISON_OK else "MISMATCH, see the comparison cell"} |
| status | {status} |
| runtime | fast {FAST["elapsed_s"]:.0f}s, full {FULL["elapsed_s"]:.0f}s |
| recordings | {len(RECORDINGS)}, from the {len(TIER_A)} classes recorded more than once |
| rows used | {FULL["rows_loaded"]:,}, equal from every recording |
| rows read to draw them | {FULL["rows_read"]:,} |
| features | {len(FEATURES)}, after dropping {", ".join(DROPPED)} |
| model | RandomForest, {N_ESTIMATORS} trees, min leaf {MIN_SAMPLES_LEAF}, {TEST_FRACTION:.0%} held out |
| capture identification, all {len(FEATURES)} features (chance {all_features["chance"]:.4f}) | {all_features["accuracy"]:.4f} accuracy, {all_features["macro_f1"]:.4f} macro-F1 |
| capture identification, attack class held fixed | **{FULL["within_class_mean_accuracy"]:.4f}** mean accuracy over {len(TIER_A)} classes, chance {FULL["within_class_mean_chance"]:.4f} |
| timing family only, {len(FAMILIES["timing"])} features | {lines["capture_family_timing"]["accuracy"]:.4f} |
| protocol family, {len(FAMILIES["protocol"])} features | {lines["capture_family_protocol"]["accuracy"]:.4f} |
| statistical family, {len(FAMILIES["statistical"])} features | {lines["capture_family_statistical"]["accuracy"]:.4f} |
| the five features named by published work | {lines["capture_prior_five"]["accuracy"]:.4f} |
| the three timing features among them | {lines["capture_timing_three"]["accuracy"]:.4f} |
| attack macro-F1, whole recording held out | {FULL["attack_macro_f1_held_out"]:.4f} |
| attack macro-F1, rows pooled | {FULL["attack_macro_f1_pooled"]:.4f} |
| difference, pooled minus held out | {FULL["attack_macro_f1_difference"]:.4f} |
| largest between-recording variance ratio, attack fixed | {top_ratio["feature"]}, median {top_ratio["median_ratio"]:,.3f}, {top_ratio["max_ratio"]:,.3f} on {top_ratio["worst_class"]} |
| recording identifiable | {verdict["recording_identifiable"]} |
| artefacts | {FULL["out_dir"]}, holding variance_ranking.json, NB03_verdict.json, {len(FULL["figures"])} figures and {len(FULL["run_names"])} runs |

Per-class capture identification, attack held fixed: {" · ".join(f"{row.scope.replace('within ', '')} {row.accuracy:.4f} (chance {row.chance:.3f})" for row in FULL["within_class_table"].itertuples())}

{verdict["consequence"]}
'''

print(ledger)